In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
from IPython.display import display

In [ ]:
def get_apartment_links(page_url):
    """Збирає посилання на всі квартири з однієї сторінки пошуку."""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    response = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    links = set()
    elements = soup.find_all(attrs={"data-event-options": re.compile(r"page_id:\d+")})
    for el in elements:
        match = re.search(r'page_id:(\d+)', el.get('data-event-options', ''))
        if match:
            links.add(f"https://lun.ua/realty/{match.group(1)}")
            
    return list(links)


def parse_apartment_data(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Структура БЕЗ building_details, але З гео-координатами та районом
    data = {
        'url': url, 'price': None, 'currency': None, 'address': None, 'rooms': None, 
        'area_total': None, 'area_living': None, 'area_kitchen': None,
        'floor': None, 'total_floors': None, 
        'build_year': None, 'construction_tech': None, 'heating_type': None,
        'lat': None, 'lon': None, 'district': None, # Гео-дані з ЛУН
        'commission': None, 'description': None
    }
    
    tech_mapping = {
        'моноліт': 'монолітно-каркасна',
        'утеплена панель': 'утеплена панель',
        'панель': 'панельна технологія',
        'цегл': 'цегляна технологія',
        'блок': 'блочна технологія',
        'блочн': 'блочна технологія'
    }
    
    # 1. ЦІНА ТА ВАЛЮТА
    raw_price = None
    price_tag = soup.find(class_=re.compile("RealtyDetails_priceMain|RealtyCard_price"))
    if price_tag:
        raw_price = price_tag.text.strip()
    else:
        og_title = soup.find("meta", property="og:title")
        title_text = og_title["content"] if og_title else (soup.title.text if soup.title else "")
        if title_text:
            price_match = re.search(r'((?:[\$€]\s*)?[\d\s\u202f\u200a]+(?:грн|\$|€)?)', title_text)
            if price_match: 
                raw_price = price_match.group(1).strip()

    if raw_price:
        raw_price_lower = raw_price.lower()
        if '$' in raw_price_lower: data['currency'] = 'USD'
        elif '€' in raw_price_lower: data['currency'] = 'EUR'
        elif 'грн' in raw_price_lower: data['currency'] = 'UAH'
            
        clean_price = re.sub(r'[^\d]', '', raw_price)
        if clean_price: data['price'] = clean_price

    # 2. АДРЕСА
    address_tag = soup.find(class_=re.compile("RealtyDetails_address|Realty_address"))
    if address_tag: data['address'] = address_tag.text.strip()

    # 3. ОПИС
    desc_tag = soup.find("article") or soup.find(attrs={"itemprop": "description"}) or soup.find(class_=re.compile("description|Description|ExpandableText_text"))
    if desc_tag: data['description'] = desc_tag.text.strip()
    
    # 4. КОМІСІЯ (Лейбли)
    labels = soup.find_all(class_=re.compile("LabelRefresh-module_content|LabelList_label"))
    for label in labels:
        if 'без комісії' in label.text.lower():
            data['commission'] = 'без комісії'
            break

    # 5. ХАРАКТЕРИСТИКИ
    temp_details = [] # тимчасовий список для перевірки дублікатів
    properties = soup.find_all(class_=re.compile("PropertyItem|Characteristic|param"))
    for prop in properties:
        text = prop.text.strip()
        if not text or text in temp_details: continue
        temp_details.append(text)
        
        text_lower = text.lower()
        
        # Комісія
        if 'комісі' in text_lower and not data['commission']:
            if 'без' in text_lower: data['commission'] = 'без комісії'
            elif '%' in text_lower or re.search(r'\d+', text_lower): data['commission'] = text
        
        # Кімнати
        if 'кімнат' in text_lower and not data['rooms']: 
            rooms_match = re.search(r'\d+', text)
            if rooms_match: data['rooms'] = rooms_match.group(0)
                
        # Площа
        elif 'м²' in text and not data['area_total']: 
            area_clean = text.replace('м²', '').strip()
            area_parts = [p.strip() for p in area_clean.split('/')]
            if len(area_parts) >= 1: data['area_total'] = area_parts[0]
            if len(area_parts) >= 2: data['area_living'] = area_parts[1]
            if len(area_parts) >= 3: data['area_kitchen'] = area_parts[2]
                
        # Поверх
        elif 'поверх' in text_lower and not data['floor']:
            floor_match = re.search(r'поверх (\d+) з (\d+)', text)
            if floor_match:
                data['floor'] = floor_match.group(1)
                data['total_floors'] = floor_match.group(2)
            else:
                data['floor'] = text
                
        # Рік будівництва
        elif 'рік будівництва' in text_lower or 'рік побудови' in text_lower:
            year_match = re.search(r'\d{4}', text)
            if year_match: data['build_year'] = year_match.group(0)
                
        # Тип опалення 
        elif 'опалення' in text_lower and 'без світла' not in text_lower:
            data['heating_type'] = text
            
        # Технологія будівництва
        if not data['construction_tech']:
            for key, exact_value in tech_mapping.items():
                if key in text_lower:
                    if key == 'панель' and 'утеплена панель' in text_lower: continue 
                    data['construction_tech'] = exact_value
                    break

    # 6. Резервний пошук в описі
    if data['description']:
        desc_lower = data['description'].lower()
        if not data['commission']:
            comm_match = re.search(r'(?i)(?:комісі|комісійні).*?(\d+\s*%|без комісії)', data['description'])
            if comm_match: data['commission'] = comm_match.group(0)
                
        if not data['construction_tech']:
            for key, exact_value in tech_mapping.items():
                if key in desc_lower:
                    if key == 'панель' and 'утеплена панель' in desc_lower: continue 
                    data['construction_tech'] = exact_value
                    break

    # 7. ГЕО-ДАНІ (Координати та Район безпосередньо з коду ЛУН)
    html_str = response.text
    
    # ЛУН записує координати як "location":[longitude, latitude]
    location_match = re.search(r'(?:\\"|")location(?:\\"|")\s*:\s*\[\s*([\d\.]+)\s*,\s*([\d\.]+)\]', html_str)
    if location_match:
        data['lon'] = location_match.group(1)
        data['lat'] = location_match.group(2)
        
    # Район визначається через супутню геолокаційну сутність
    district_match = re.search(r'(?:\\"|")name(?:\\"|")\s*:\s*(?:\\"|")([^"\\]+)(?:\\"|"),\s*(?:\\"|")realtyGeoUrl(?:\\"|").*?-district', html_str)
    if district_match:
        data['district'] = district_match.group(1)

    # 8. ОЧИЩЕННЯ ТЕКСТУ ВІД ПЕРЕНОСІВ (БЕЗ building_details)
    for key in ['address', 'description', 'commission', 'heating_type']:
        if data[key]:
            clean_text = re.sub(r'[\n\r\t]+', ' ', data[key])
            data[key] = re.sub(r'\s+', ' ', clean_text).strip()
            
    return data

def run_scraper(pages_list, limit_flats=None):
    """
    Головна функція. Приймає список сторінок і повертає готовий DataFrame.
    limit_flats - обмежує кількість зібраних квартир для тестів (напр. 5).
    """
    all_links = []
    print("Шукаємо посилання на квартири...")
    for page in pages_list:
        links = get_apartment_links(page)
        all_links.extend(links)
        print(f"Знайдено {len(links)} на {page}")
        time.sleep(0.1) 
        
    unique_links = list(set(all_links))
    if limit_flats:
        unique_links = unique_links[:limit_flats]
        
    print(f"\nПочинаємо збір даних по {len(unique_links)} квартирах...")
    scraped_data = []
    
    for i, link in enumerate(unique_links):
        print(f"[{i+1}/{len(unique_links)}] {link}")
        scraped_data.append(parse_apartment_data(link))
        time.sleep(0.3)
        
    print("\nГотово!")
    return pd.DataFrame(scraped_data)

In [ ]:

# ==========================================
# ЗАПУСК КОДУ: ПАРСИНГ ІЗ ЗБЕРЕЖЕННЯМ КОЖНІ 300
# ==========================================
base_url = "https://lun.ua/rent/kyiv/flats"
output_csv = "kyiv_flats_data.csv"
save_interval = 300

# 1. Перевіряємо, чи є вже збережені дані
scraped_urls = set()
if os.path.exists(output_csv):
    existing_df = pd.read_csv(output_csv)
    scraped_urls = set(existing_df['url'].tolist())
    print(f"Знайдено файл збереження. Вже зібрано квартир: {len(scraped_urls)}")

# 2. Шукаємо всі посилання на всіх сторінках
all_links = []
page = 1
print("Шукаємо посилання на сайті...")

while True:
    url = base_url if page == 1 else f"{base_url}?page={page}"
    print(f"Перевіряю сторінку {page}...", end='\r')
    
    links = get_apartment_links(url)
    if not links:
        print(f"\nКвартири закінчились на сторінці {page-1}.")
        break
        
    all_links.extend(links)
    page += 1
    time.sleep(0.4)

unique_links = list(set(all_links))
print(f"\nВсього знайдено посилань на сайті: {len(unique_links)}")

# 3. Відфільтровуємо тільки нові квартири
links_to_scrape = [link for link in unique_links if link not in scraped_urls]
print(f"Нових квартир для парсингу: {len(links_to_scrape)}")

# 4. Парсимо і зберігаємо порціями
batch_data = []

# Для швидкого тесту можеш розкоментувати рядок нижче:
# links_to_scrape = links_to_scrape[:5] 

for i, link in enumerate(links_to_scrape):
    print(f"[{i+1}/{len(links_to_scrape)}] {link}")
    
    try:
        flat_data = parse_apartment_data(link)
        batch_data.append(flat_data)
    except Exception as e:
        print(f"Помилка при парсингу {link}: {e}")
        
    time.sleep(0.5)
    
    # Автозбереження кожні 300 або якщо це остання квартира в списку
    is_last = (i == len(links_to_scrape) - 1)
    if len(batch_data) == save_interval or is_last:
        df_batch = pd.DataFrame(batch_data)
        
        # Якщо файлу нема - пишемо з заголовками, якщо є - дописуємо (append)
        file_exists = os.path.exists(output_csv)
        df_batch.to_csv(output_csv, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
        
        print(f"\n[АВТОЗБЕРЕЖЕННЯ] {len(batch_data)} нових квартир додано у {output_csv}!\n")
        batch_data = [] # Очищаємо порцію для наступних

print("\nЗбір даних завершено!")

# Щоб подивитись результат прямо в Jupyter:
if os.path.exists(output_csv):
    display(pd.read_csv(output_csv).head())

Шукаємо посилання на сайті...


KeyboardInterrupt: 